# 07. Yurak kasalliklari xavfini bashorat qilish (Heart Disease Risk Prediction)
### Kengaytirilgan va Chuqur Optimallashtirilgan Machine Learning Pipeline
**Texnologiyalar:** Python, Pandas, Scikit-learn (KNeighborsClassifier, SVC, Stratified 5-Fold Cross-Validation, GridSearchCV, ROC/PR egri chiziqlari, XAI tahlili)

Ushbu daftarda UCI Cleveland Heart Disease ma'lumotlar to'plami asosida bemorlarning 13 ta klinik ko'rsatkichi tahlil qilinib, yurak xastaligi xavfini bashorat qiluvchi **KNN** va **SVM** modellarining chuqur optimallashtirilgan arxitekturasi yaratilgan.

## 1. Kerakli kutubxonalarni yuklash va ma'lumotlarni import qilish

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score
)

print("Barcha kutubxonalar muvaffaqiyatli yuklandi!")

In [2]:
# Datasetni yuklash va tozalash
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip().str.replace("﻿", "")
duplicates = df.duplicated().sum()
if duplicates > 0:
    df.drop_duplicates(inplace=True)
print(f"Dataset o'lchami: {df.shape[0]} ta qator, {df.shape[1]} ta ustun")
df.head()

## 2. Dastlabki Ma'lumotlar Tahlili (EDA) va Korrelyatsiya

In [3]:
# Maqsadli o'zgaruvchi taqsimoti
print("Klasslar taqsimoti:")
print(df["target"].value_counts(normalize=True).map(lambda n: f"{n*100:.1f}%"))

# Xususiyatlarning yurak kasalligi bilan korrelyatsiyasi (Pearson r)
corrs = df.corr()["target"].drop("target").sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=corrs.values, y=corrs.index, palette="coolwarm")
plt.title("Klinik Xususiyatlarning Yurak Kasalligi Bilan Korrelyatsiyasi")
plt.xlabel("Pearson Korrelyatsiya Koeffitsiyenti (r)")
plt.show()

## 3. Train / Test (80/20) Stratified Split va StandardScaler

In [4]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X)

print(f"O'qitish to'plami: {X_train.shape[0]} ta namuna")
print(f"Sinov to'plami:   {X_test.shape[0]} ta namuna")

## 4. 5-Fold Stratified Kross-Validatsiya (Keng Qamrovli Baholash)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_cv_acc = cross_val_score(KNeighborsClassifier(n_neighbors=7, weights="distance"), X_all_scaled, y, cv=cv, scoring="accuracy")
svm_cv_acc = cross_val_score(SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42), X_all_scaled, y, cv=cv, scoring="accuracy")

print("=== 5-Fold Kross-Validatsiya Aniqligi ===")
print(f"KNN O'rtacha: {knn_cv_acc.mean()*100:.2f}% (±{knn_cv_acc.std()*100:.2f}%)")
print(f"SVM O'rtacha: {svm_cv_acc.mean()*100:.2f}% (±{svm_cv_acc.std()*100:.2f}%)")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), knn_cv_acc*100, marker="o", label="KNN (k=7)")
plt.plot(range(1, 6), svm_cv_acc*100, marker="s", label="SVM (RBF)")
plt.title("5-Fold Kross-Validatsiya Aniqligi Har Bir Foldda")
plt.xlabel("Fold")
plt.ylabel("Aniqlik (%)")
plt.legend()
plt.grid(True)
plt.show()

## 5. Giperparametrlarni Optimallashtirish va Modellar O'qitish

In [6]:
# KNN modeli (k=7)
knn = KNeighborsClassifier(n_neighbors=7, weights="distance")
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

# SVM modeli (RBF Kernel)
svm = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("=== Model Metriklari Taqqoslash ===")
print(f"KNN -> Accuracy: {accuracy_score(y_test, y_pred_knn)*100:.2f}%, Recall: {recall_score(y_test, y_pred_knn)*100:.2f}%, F1: {f1_score(y_test, y_pred_knn)*100:.2f}%")
print(f"SVM -> Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%, Recall: {recall_score(y_test, y_pred_svm)*100:.2f}%, F1: {f1_score(y_test, y_pred_svm)*100:.2f}%")

## 6. ROC Egri Chiziqlari va Confusion Matrix Tahlili

In [7]:
fpr_k, tpr_k, _ = roc_curve(y_test, y_prob_knn)
fpr_s, tpr_s, _ = roc_curve(y_test, y_prob_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_s, tpr_s, "r-", label=f"SVM (AUC = {roc_auc_score(y_test, y_prob_svm)*100:.1f}%)", linewidth=2)
plt.plot(fpr_k, tpr_k, "b--", label=f"KNN (AUC = {roc_auc_score(y_test, y_prob_knn)*100:.1f}%)", linewidth=2)
plt.plot([0, 1], [0, 1], "k:", label="Tasodifiy chegara")
plt.title("ROC Egri Chizig'i (ROC Curve)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.legend()
plt.grid(True)
plt.show()

## 7. Xulosa va Klinik Tavsiyalar
1. **SVM (RBF Kernel)** bemorlarda yurak xastaligini aniqlashda eng yuqori aniqlik (**83.61%**) va eng muhimi eng yuqori **Recall (93.94%)** ko'rsatkichini qayd etdi (33 ta kasallangan bemordan 31 tasini aniq topdi).
2. 5-Fold Stratified Kross-Validatsiya har ikkala modelning umumiy barqarorligini tasdiqladi (**81.09%** SVM va **81.07%** KNN).
3. Barcha parametrlar, ROC koordinatalari, K-egrisi va xususiyatlar og'irliklari Vercel interaktiv tizimi uchun  ga to'liq eksport qilindi.